# Demo: pronóstico de las próximas 8 semanas y medicamentos

Este notebook carga los pesos ya entrenados de la LSTM de dengue, pronostica los casos de cada provincia a partir de
la última semana que hay en los datos y convierte esos casos en medicamentos con el kit por caso
(`data/external/kit_medicamentos.csv`).

In [1]:
import os, sys, glob, pathlib
EN_COLAB = "google.colab" in sys.modules
REPO = "https://github.com/Kysard/dengue-costa-norte.git"   # repositorio del grupo
if EN_COLAB and not glob.glob("**/src/datos.py", recursive=True):
    # Opción A: clonar el repositorio del grupo
    !git clone -q {REPO}
    # Opción B: si no se pudo clonar, subir el zip del proyecto
    if not glob.glob("**/src/datos.py", recursive=True):
        from google.colab import files
        for nombre in files.upload():
            !unzip -q -o "{nombre}"
candidatas = [pathlib.Path("."), pathlib.Path("..")] + [pathlib.Path(p).parents[1] for p in glob.glob("**/src/datos.py", recursive=True)]
raiz = next(c.resolve() for c in candidatas if (c / "src" / "datos.py").exists())
os.chdir(raiz)
sys.path.insert(0, "src")
print("Proyecto listo:", raiz.name)

Proyecto listo: proyecto_grupo5


In [2]:
import glob
import pandas as pd
import experimentos as X
from inferencia import pronosticar, medicamentos_dengue

panel = X.cargar_panel()
pesos = sorted(glob.glob("results/pesos/dengue_lstm_s*.pt"))
print("Pesos disponibles:", pesos)

Pesos disponibles: ['results/pesos/dengue_lstm_s0.pt', 'results/pesos/dengue_lstm_s1.pt', 'results/pesos/dengue_lstm_s2.pt', 'results/pesos/dengue_lstm_s3.pt', 'results/pesos/dengue_lstm_s4.pt']


Si no aparecen pesos en `results/pesos/`, se pueden descargar del enlace del README o volver a generar corriendo el notebook principal.

In [3]:
pron, info = pronosticar(panel, "dengue", pesos[0])
print("Última semana con datos:", info["ultima_semana"], "| modelo:", info["modelo"], "| semilla:", info["semilla"])
tabla = pron.pivot_table(index="provincia", columns="semana_inicia", values="casos_pronosticados").round(0)
tabla

Última semana con datos: 2025-SE53 | modelo: lstm | semilla: 0


semana_inicia,2026-01-04,2026-01-11,2026-01-18,2026-01-25,2026-02-01,2026-02-08,2026-02-15,2026-02-22
provincia,,,,,,,,
AYABACA,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
CHICLAYO,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
CONTRALMIRANTE VILLAR,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
FERREÑAFE,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
HUANCABAMBA,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
LAMBAYEQUE,1.0,1.0,1.0,1.0,1.0,1.0,1.0,2.0
MORROPON,1.0,1.0,1.0,1.0,1.0,1.0,2.0,2.0
PAITA,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
PIURA,2.0,2.0,3.0,3.0,3.0,3.0,3.0,4.0


In [4]:
kit = pd.read_csv("data/external/kit_medicamentos.csv")
med = medicamentos_dengue(pron, panel, kit)
med.set_index("provincia").drop(columns="ubigeo_prov").round(0).sort_values("casos_pronosticados", ascending=False)

,casos_pronosticados,Cloruro de sodio 0.9% 1 L,Equipo de venoclisis,Paracetamol 500 mg,Sales de rehidratacion oral
provincia,,,,,
TUMBES,50.0,25.0,6.0,946.0,218.0
PIURA,23.0,11.0,3.0,438.0,101.0
MORROPON,10.0,5.0,1.0,200.0,47.0
LAMBAYEQUE,10.0,4.0,1.0,200.0,47.0
SULLANA,2.0,1.0,0.0,47.0,11.0
TALARA,2.0,1.0,0.0,32.0,7.0
ZARUMILLA,2.0,1.0,0.0,47.0,11.0
AYABACA,1.0,0.0,0.0,15.0,4.0
CHICLAYO,1.0,0.0,0.0,16.0,4.0


Son los totales de las próximas 8 semanas por provincia. Para repartir los casos por gravedad usamos la proporción que tuvo cada provincia desde 2015.